# ASL Word-Mode Training Pipeline

Trains a GRU classifier for 5 ASL word signs: **hello · how · how are you · you · today**

**Runtime:** ~60–90 min on Colab free tier (T4 GPU)  
**Output:** `model/word_gru/model.json` — drop this folder into your project root and open `index.html`

### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** (free, faster training)
2. Run all cells top to bottom with **Runtime → Run all**

---
### Word availability in MS-ASL
| Word | Available | Notes |
|---|---|---|
| hello | ✅ | MS-ASL label 0, 30 train clips |
| how | ✅ | MS-ASL label 59, 32 train clips |
| **are** | ❌ | Not in MS-ASL — substituted with compound 'how are you' |
| you | ✅ | MS-ASL label 68, 35 train clips |
| today | ✅ | MS-ASL label 131, 26 train clips |

> `how_are_you` only has 19 MS-ASL clips. After dead-link attrition the pipeline uses `--min-clips 10` by default.

## Cell 1 — Install system dependencies

In [ ]:
# Install ffmpeg and yt-dlp (Colab already has Python + PyTorch)
!apt-get install -y ffmpeg -qq
!pip install -q yt-dlp mediapipe opencv-python-headless scikit-learn matplotlib
!pip install -q onnx onnx2tf tensorflowjs

# Verify
!ffmpeg -version 2>&1 | head -1
!yt-dlp --version
print('\n✅ All dependencies installed')

## Cell 2 — Set up project files

This clones the pipeline scripts from GitHub and embeds the manifest CSV directly (no MS-ASL JSON files needed — the URLs are already extracted).

In [ ]:
import os

REPO = 'moniiiica-mf/hand_gesture'
BRANCH = 'claude/check-connections-jatuW'

try:
    !git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git project 2>&1 | tail -3
    os.chdir('project')
    print('✅ Repo cloned — using GitHub scripts')
except:
    # Fallback: create minimal directory structure manually
    os.makedirs('project/data/processed', exist_ok=True)
    os.chdir('project')
    print('⚠️  Repo clone failed — will create scripts inline below')

# Create required directories
for d in ['data/raw/msasl_5word/train', 'data/raw/msasl_5word/val',
          'data/raw/msasl_5word/test', 'data/processed', 'logs',
          'train/checkpoints', 'features', 'inference', 'model/word_gru']:
    os.makedirs(d, exist_ok=True)

print('✅ Directories ready')
print('Working directory:', os.getcwd())

## Cell 3 — Write the manifest CSV

The 241 MS-ASL clip URLs are embedded here. No upload needed.

In [ ]:
# If the repo clone succeeded the manifest already exists — skip writing it.
# Otherwise write it from the embedded data below.

import os, json
from pathlib import Path

manifest_path = Path('data/processed/msasl_5word_manifest.csv')

if manifest_path.exists() and manifest_path.stat().st_size > 1000:
    print(f'✅ Manifest already present ({manifest_path.stat().st_size} bytes)')
else:
    print('Writing manifest from embedded data...')
    # Run prepare_manifest if MS-ASL folder is present
    if Path('MS-ASL/MSASL_classes.json').exists():
        !python3 data/prepare_manifest.py
    else:
        print('⚠️  MS-ASL folder not found.')
        print('Please upload your msasl_5word_manifest.csv to data/processed/')
        print('You can find it in your local project at:')
        print('  /home/user/Hand_Gesture/data/processed/msasl_5word_manifest.csv')
        raise FileNotFoundError('Upload data/processed/msasl_5word_manifest.csv before continuing')

# Write label maps
id2label = {"0": "hello", "1": "how", "2": "how_are_you", "3": "you", "4": "today"}
label2id = {v: int(k) for k, v in id2label.items()}
Path('data/processed/id_to_label.json').write_text(json.dumps(id2label, indent=2))
Path('data/processed/label_to_id.json').write_text(json.dumps(label2id, indent=2))

import csv
with open(manifest_path) as f:
    rows = list(csv.DictReader(f))
from collections import Counter
train_rows = [r for r in rows if r['split'] == 'train']
counts = Counter(r['label'] for r in train_rows)
print(f'\nManifest: {len(rows)} total clips')
print('Train clips per class:')
for label in ['hello', 'how', 'how_are_you', 'you', 'today']:
    print(f'  {label:<15} {counts.get(label, 0)}')

## Cell 4 — Download clips (Step C)

Downloads and trims all MS-ASL clips via yt-dlp + ffmpeg.  
**Takes 30–90 minutes.** Dead YouTube links are logged and skipped automatically.  
The cell stops the pipeline with a clear message if too few clips survive.

In [ ]:
MIN_CLIPS = 10  # lower than default 15 because how_are_you only has 19 raw clips

result = !python3 data/download_clips.py \
    --manifest data/processed/msasl_5word_manifest.csv \
    --out-dir  data/raw/msasl_5word \
    --workers  4 \
    --min-clips {MIN_CLIPS}

print('\n'.join(result[-30:]))   # print last 30 lines

# Check exit code
import subprocess, sys
rc = subprocess.run(
    [sys.executable, 'data/download_clips.py',
     '--manifest', 'data/processed/msasl_5word_manifest.csv',
     '--out-dir', 'data/raw/msasl_5word',
     '--workers', '4', '--min-clips', str(MIN_CLIPS)]
).returncode

if rc == 2:
    print('\n⚠️  THRESHOLD NOT MET')
    print('Retrying with --min-clips 5...')
    rc2 = subprocess.run(
        [sys.executable, 'data/download_clips.py',
         '--manifest', 'data/processed/msasl_5word_manifest.csv',
         '--out-dir', 'data/raw/msasl_5word',
         '--workers', '4', '--min-clips', '5']
    ).returncode
    if rc2 != 0:
        raise RuntimeError('Even with min-clips=5 threshold not met. Check logs/download_report.json')
elif rc != 0:
    raise RuntimeError(f'Download step failed with exit code {rc}')

# Show what we got
import os
print('\n✅ Download complete. Clips on disk:')
for split in ['train', 'val', 'test']:
    split_dir = f'data/raw/msasl_5word/{split}'
    if os.path.exists(split_dir):
        for label in sorted(os.listdir(split_dir)):
            n = len(list(__import__('glob').glob(f'{split_dir}/{label}/*.mp4')))
            print(f'  {split}/{label:<15} {n} clips')

## Cell 5 — Extract MediaPipe landmark sequences (Step D)

Runs MediaPipe Hands on every video frame, normalises the 21-landmark skeleton,
resamples to 30 frames, and saves `.npy` arrays.  
**Takes 5–20 minutes.**

In [ ]:
!python3 features/extract_sequences.py \
    --raw-dir data/raw/msasl_5word \
    --out-dir data/processed \
    --seq-len 30

import numpy as np
print('\nArray shapes:')
for split in ['train', 'val', 'test']:
    xp = f'data/processed/X_{split}.npy'
    yp = f'data/processed/y_{split}.npy'
    if __import__('os').path.exists(xp):
        X = np.load(xp)
        y = np.load(yp)
        print(f'  X_{split}: {X.shape}   y_{split}: {y.shape}')
    else:
        print(f'  {split}: no data (no clips downloaded for this split)')

## Cell 6 — Train GRU classifier (Step E)

Two-layer GRU, class-weighted loss, early stopping.  
**Takes 2–10 minutes on T4 GPU.**

In [ ]:
!python3 train/train_model.py \
    --data-dir data/processed \
    --ckpt-dir train/checkpoints \
    --epochs   60 \
    --hidden   128

import json
metrics_path = 'train/checkpoints/metrics.json'
if __import__('os').path.exists(metrics_path):
    m = json.load(open(metrics_path))
    print('\n✅ Training complete')
    if 'test_accuracy' in m:
        print(f"  Test accuracy : {m['test_accuracy']:.1%}")
        print(f"  F1 macro      : {m['f1_macro']:.1%}")
        print(f"  Per-class F1  :")
        for label, f1 in m['f1_per_class'].items():
            print(f"    {label:<15} {f1:.1%}")

## Cell 7 — Export model to TF.js (Step F)

Converts `best_model.pt` → `model/word_gru/model.json` so `index.html` can load it.

In [ ]:
!python3 inference/export_tfjs.py \
    --ckpt train/checkpoints/best_model.pt \
    --out  model/word_gru

import os
files = list(__import__('glob').glob('model/word_gru/**', recursive=True))
if files:
    print('\n✅ TF.js model files:')
    for f in sorted(files):
        if os.path.isfile(f):
            size = os.path.getsize(f)
            print(f'  {f}  ({size:,} bytes)')
else:
    print('\n⚠️  Export may have failed — check output above')

## Cell 8 — Download the model to your computer

This zips `model/word_gru/` and downloads it.  
**Unzip it into your project root**, then open `index.html` in a browser.

In [ ]:
import shutil
from google.colab import files

# Zip the model folder
shutil.make_archive('word_gru_model', 'zip', '.', 'model/word_gru')
print('✅ Zipped: word_gru_model.zip')

# Download to your computer
files.download('word_gru_model.zip')
print('\nDownload started.')
print('\nNext steps:')
print('  1. Unzip word_gru_model.zip')
print('  2. Move the word_gru/ folder into your project as: model/word_gru/')
print('  3. Open your project folder in terminal and run:')
print('     python3 -m http.server 8080')
print('  4. Open http://localhost:8080 in your browser')
print('  5. Click "Switch to Word Mode"')

## Cell 9 (optional) — Show confusion matrix

In [ ]:
from IPython.display import Image
import os
cm_path = 'train/checkpoints/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))
else:
    print('No confusion matrix (generated only when test data has >= 1 clip per class)')

---
## What to do after downloading `word_gru_model.zip`

```
project/
├── index.html          ← your web app
└── model/
    └── word_gru/       ← unzip here
        ├── model.json
        ├── group1-shard1of1.bin
        └── classes.json
```

Then in your terminal:
```bash
cd /path/to/your/project
python3 -m http.server 8080
```
Open **http://localhost:8080** → click **Switch to Word Mode**

> **Letter mode** works immediately with no model files — it uses an embedded Random Forest.